# Build Team Ops Index Once (Local Laptop)

Use this notebook once to create embeddings from files in `data/` and persist the index locally.
Then load the saved index in your demo notebook without rebuilding embeddings.

## 1) Local Prerequisites

Before running this notebook, ensure:
- Ollama is installed on your laptop
- Ollama is running (or let the next cell start it)
- Your team documents are already in `data/`

Optional GPU acceleration depends on your local hardware and Ollama setup.

In [1]:
# Install Python packages
%pip install -q jedi pypdf llama-index llama-index-llms-ollama llama-index-embeddings-ollama

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Verify Ollama CLI is available (PATH or common macOS locations)
import os
import shutil

candidates = [
    shutil.which("ollama"),
    "/opt/homebrew/bin/ollama",
    "/usr/local/bin/ollama",
    os.path.expanduser("~/.ollama/bin/ollama"),
]

OLLAMA_CMD = next((p for p in candidates if p and os.path.exists(p)), None)

if OLLAMA_CMD is None:
    raise RuntimeError(
        "Ollama CLI not found. This is system-level (not a Python venv package). "
        "Install Ollama from https://ollama.com/download or run 'brew install ollama', "
        "then restart VS Code/Jupyter and rerun."
    )

print(f"Ollama CLI found: {OLLAMA_CMD}")

Ollama CLI found: /opt/homebrew/bin/ollama


In [3]:
# Start Ollama in background if needed
import os
import subprocess
import time
import urllib.request

os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'

def ollama_api_up(url='http://127.0.0.1:11434/api/tags'):
    try:
        with urllib.request.urlopen(url, timeout=2) as _r:
            return True
    except Exception:
        return False

if not ollama_api_up():
    subprocess.Popen([OLLAMA_CMD, 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

if not ollama_api_up():
    raise RuntimeError('Ollama server is not reachable at http://127.0.0.1:11434')

print('Ollama server is running.')

Ollama server is running.


In [4]:
# Pull generation and embedding models (safe to rerun)
import subprocess

subprocess.run([OLLAMA_CMD, 'pull', 'llama3.2:3b'], check=True)
subprocess.run([OLLAMA_CMD, 'pull', 'nomic-embed-text'], check=True)
print('Models are ready.')

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ 

Models are ready.


pulling manifest 
pulling 970aa74c0a90: 100% ▕██████████████████▏ 274 MB                         
pulling c71d239df917: 100% ▕██████████████████▏  11 KB                         
pulling ce4a164fc046: 100% ▕██████████████████▏   17 B                         
pulling 31df23ea7daa: 100% ▕██████████████████▏  420 B                         
verifying sha256 digest 
writing manifest 
success 


## 2) Use Local Team Data Folder (`data/`)

In [6]:
from pathlib import Path
import shutil
import zipfile
import re

data_dir = Path('data')
if not data_dir.exists():
    raise FileNotFoundError("Local folder 'data/' not found.")

# Auto-extract ZIPs in data/ to data/<zip_name>/ if not already extracted.
zip_files = list(data_dir.glob('*.zip'))
if zip_files:
    print('ZIP files detected in data/:')
    for z in zip_files:
        extract_dir = data_dir / z.stem
        if not extract_dir.exists():
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(extract_dir)
            print(f"- Extracted {z.name} -> {extract_dir}")
        else:
            print(f"- Already extracted: {z.name} -> {extract_dir}")

# Remove unneeded archive folder(s) after unzip to speed up indexing.
removed_dirs = []
for p in data_dir.rglob('*'):
    if p.is_dir() and p.name in {'Archived Files', 'SArchived Files'}:
        shutil.rmtree(p)
        removed_dirs.append(str(p))

if removed_dirs:
    print('Removed archive folder(s):')
    for d in removed_dirs:
        print('-', d)

def normalize_name(name: str) -> str:
    return re.sub(r'[^a-z0-9]', '', name.lower())

# Start with only these folders for a faster first index build.
selected_subdirs = ['Timesheets', 'Non-Cost-Recovered Time', 'Projects']
all_dirs = [p for p in data_dir.rglob('*') if p.is_dir()]

selected_roots = []
for desired in selected_subdirs:
    wanted = normalize_name(desired)
    matches = []
    for d in all_dirs:
        name_norm = normalize_name(d.name)
        rel_norm = normalize_name(str(d.relative_to(data_dir)))
        if wanted in name_norm or wanted in rel_norm:
            matches.append(d)

    if matches:
        # Prefer shallowest matching path.
        matches = sorted(matches, key=lambda x: len(x.parts))
        selected_roots.append(matches[0])

# Remove duplicates while preserving order.
unique_roots = []
seen = set()
for r in selected_roots:
    if str(r) not in seen:
        unique_roots.append(r)
        seen.add(str(r))
selected_roots = unique_roots

if not selected_roots:
    print('Directory structure under data/:')
    for p in sorted(all_dirs, key=lambda x: str(x))[:120]:
        print('-', p)
    raise ValueError(
        f"Could not find any of the selected folders: {selected_subdirs}. "
        "Check printed directory structure for exact names."
    )

all_files = []
for root in selected_roots:
    all_files.extend([p for p in root.rglob('*') if p.is_file()])

if not all_files:
    raise ValueError('Selected folders were found, but no files were found inside them.')

print(f"Using data directory: {data_dir.resolve()}")
print('Indexing only these folders:')
for root in selected_roots:
    print('-', root)
print(f"Found {len(all_files)} files in selected folders")

ZIP files detected in data/:
- Extracted Technicals.zip -> data/Technicals
- Extracted Operations.zip -> data/Operations
Removed archive folder(s):
- data/Operations/Operations/Timesheets/Archived Files
- data/Operations/Operations/Non-Cost-Recovered Time/Archived Files
Using data directory: /Users/c.russon/Desktop/rse_role/technical-llamaindex/data
Indexing only these folders:
- data/Operations/Operations/Timesheets
- data/Operations/Operations/Non-Cost-Recovered Time
- data/Operations/Operations/Projects
Found 44 files in selected folders


In [7]:
# Optional preview of files that will be indexed
for p in all_files[:40]:
    print('-', p)
if len(all_files) > 40:
    print('...')

- data/Operations/Operations/Timesheets/RSA Project Finance Codes.xlsx
- data/Operations/Operations/Timesheets/Timesheet Submission Screen Recording.mov
- data/Operations/Operations/Timesheets/Timesheet FAQs.docx
- data/Operations/Operations/Timesheets/Timesheet_categorisation_guidance.xlsx
- data/Operations/Operations/Timesheets/Standardised PS Timesheet Template (RSA).xlsx
- data/Operations/Operations/Timesheets/Non-Cost-Recovered-Time Meeting Recording - 20250903.mp4
- data/Operations/Operations/Timesheets/Standardised PS Timesheet Template (RSA) - Example.xlsx
- data/Operations/Operations/Timesheets/Docusign.url
- data/Operations/Operations/Non-Cost-Recovered Time/Pump Priming Brief.docx
- data/Operations/Operations/Non-Cost-Recovered Time/Personal Development Ideas Wall.url
- data/Operations/Operations/Non-Cost-Recovered Time/Non-Cost-Recovered Time.pptx
- data/Operations/Operations/Non-Cost-Recovered Time/Non-Cost-Recovered-Time Meeting Recording - 20250903.mp4
- data/Operations/

## 3) Build Embeddings + Persist Index Locally

This is the slow step. Run once, then reuse the saved index folder.

In [8]:
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

Settings.llm = Ollama(model='llama3.2:3b', request_timeout=180.0)
Settings.embed_model = OllamaEmbedding(model_name='nomic-embed-text')
Settings.chunk_size = 512
Settings.chunk_overlap = 50

documents = []
for root in selected_roots:
    docs = SimpleDirectoryReader(str(root), recursive=True).load_data()
    print(f'Loaded {len(docs)} docs from {root.name}')
    documents.extend(docs)

print(f'Total loaded docs: {len(documents)}')

index = VectorStoreIndex.from_documents(documents)

persist_dir = 'data/team_ops_index_nomic'
index.storage_context.persist(persist_dir=persist_dir)
print(f'Persisted index to: {persist_dir}')

Loaded 8 docs from Timesheets
Loaded 5 docs from Non-Cost-Recovered Time
Loaded 31 docs from Projects
Total loaded docs: 44


KeyboardInterrupt: 

In [ ]:
# Optional quick validation query
qe = index.as_query_engine(similarity_top_k=4)
print(qe.query('How do I submit my timesheets?'))